# LUMEN AI - Smart City Object Detection Platform
## Production-Grade Google Colab Model Training Environment

This notebook coordinates the end-to-end training pipeline for fine-tuning **YOLO11** on the **LUMEN** smart city civic infrastructure dataset.

### Training Architecture Overview
```
Laptop (Git Push) ──> GitHub (Git Clone) ──> Google Colab (Tesla T4/L4/A100 GPU)
                                                    │ 
     Google Drive <─────────────────────────────────┘ (Copies dataset, saves weights, logs & metrics)
```

### Pre-requisites & Setup Steps
1. **Upload Dataset to Google Drive**: Zip or copy your `dataset/` folder so it resides on Google Drive at `MyDrive/LUMEN/dataset`.
2. **Push Code to GitHub**: Put your project code (excluding `dataset/`, `runs/`, `.env`, and virtual environments) onto your GitHub repository.
3. **Run Cells**: Execute the following cells sequentially.

### Step 1: Mount Google Drive
Mount your Google Drive to load the dataset and persist trained weights and logs dynamically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Clone/Pull Latest GitHub Repository
Enter your GitHub repository parameters to clone the project to the Colab environment. If the repository already exists, it will pull the latest updates.

In [ ]:
import os

# CONFIGURATION - Adjust with your repository information
GITHUB_USER = "YOUR_GITHUB_USERNAME" # Replace with your GitHub Username
GITHUB_REPO = "YOUR_REPOSITORY_NAME" # Replace with your Repo Name
GITHUB_BRANCH = "main"

%cd /content
repo_path = f"/content/{GITHUB_REPO}"

if not os.path.exists(repo_path):
    print(f"Cloning new repository from: https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git")
    !git clone -b {GITHUB_BRANCH} https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git
    %cd {GITHUB_REPO}
else:
    print(f"Repository folder exists at {repo_path}. Pulling latest updates...")
    %cd {repo_path}
    !git fetch --all
    !git reset --hard origin/{GITHUB_BRANCH}
    !git pull

### Step 3: Install Required Dependencies
Install the packages listed in `requirements.txt` along with the latest `ultralytics` framework.

In [ ]:
# Install project requirements and ultralytics
!pip install -r requirements.txt
!pip install ultralytics
import ultralytics
ultralytics.checks()

### Step 4: Verify GPU and CUDA Environment
Check availability, name, CUDA version, VRAM limits, and hardware compatibility (supporting Tesla T4, L4, A100, and RTX GPUs).

In [ ]:
import torch
import os

print("======================================")
print("     GPU & SYSTEM INFORMATION")
print("======================================")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available:      {cuda_available}")
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_properties = torch.cuda.get_device_properties(0)
    vram_gb = gpu_properties.total_memory / (1024**3)
    print(f"GPU Name:            {gpu_name}")
    print(f"Total VRAM:          {vram_gb:.2f} GB")
    print(f"CUDA Device Version: {torch.version.cuda}")
else:
    print("WARNING: GPU NOT DETECTED! Training will run on CPU and be extremely slow.")
print(f"PyTorch Version:     {torch.__version__}")
print(f"CPU Count:           {os.cpu_count()}")
print("======================================")

### Step 5: Copy Dataset from Google Drive & Adjust Config Paths
We copy the dataset folder from your Drive (`MyDrive/LUMEN/dataset`) to the local VM storage (`/content/dataset`) for faster training IO, and dynamically update the relative path inside `data.yaml` to ensure Ultralytics works correctly.

In [ ]:
import shutil
import yaml
import glob

drive_dataset_path = "/content/drive/MyDrive/LUMEN/dataset"
local_dataset_path = "/content/dataset"
local_yaml_path = "/content/dataset/data.yaml"

# 1. Copy dataset to local directory for quick I/O access
if not os.path.exists(drive_dataset_path):
    raise FileNotFoundError(f"Dataset not found in Google Drive at: {drive_dataset_path}. Please check your Drive setup!")

if os.path.exists(local_dataset_path):
    print("Local dataset copy already exists. Skipping copy steps.")
else:
    print("Copying dataset from Google Drive to local instance VM disk (takes 1-3 minutes)...")
    shutil.copytree(drive_dataset_path, local_dataset_path)
    print("Dataset copy successfully completed.")

# 2. Dynamically modify data.yaml relative path key to `/content/dataset` for the Colab runtime environment
if os.path.exists(local_yaml_path):
    with open(local_yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)
    
    # Overwrite the path variable to align with local Colab directory structure
    yaml_data['path'] = "/content/dataset"
    
    with open(local_yaml_path, 'w') as f:
        yaml.safe_dump(yaml_data, f)
    print("Dynamically aligned path key inside data.yaml to: /content/dataset")
else:
    print(f"WARNING: data.yaml was not found at: {local_yaml_path}. Verify your dataset folders!")

# 3. Verify files and print statistics
train_images = glob.glob(os.path.join(local_dataset_path, "images/train/*"))
val_images = glob.glob(os.path.join(local_dataset_path, "images/val/*"))
print("--------------------------------------")
print(f"Training images counted:   {len(train_images)}")
print(f"Validation images counted: {len(val_images)}")
print("--------------------------------------")

### Step 6: Sync Previous Checkpoints from Google Drive for Resuming
If training was interrupted due to Colab disconnection, we check for an existing `last.pt` checkpoint inside Google Drive (`LUMEN/checkpoints/last.pt`) and copy it back to local directories to resume from the last completed epoch automatically.

In [ ]:
drive_checkpoints_dir = "/content/drive/MyDrive/LUMEN/checkpoints"
# Align local target directory with GITHUB_REPO
local_weights_dir = f"/content/{GITHUB_REPO}/runs/train_run/weights"
os.makedirs(local_weights_dir, exist_ok=True)

drive_last_pt = os.path.join(drive_checkpoints_dir, "last.pt")
local_last_pt = os.path.join(local_weights_dir, "last.pt")
drive_best_pt = os.path.join(drive_checkpoints_dir, "best.pt")
local_best_pt = os.path.join(local_weights_dir, "best.pt")

resume_flag = False

if os.path.exists(drive_last_pt):
    print(f"Found existing training checkpoint in Drive: {drive_last_pt}")
    print("Restoring checkpoint to local environment directory to enable auto-resume...")
    shutil.copy(drive_last_pt, local_last_pt)
    if os.path.exists(drive_best_pt):
        shutil.copy(drive_best_pt, local_best_pt)
    resume_flag = True
    print("Restoration successful. Training will resume automatically.")
else:
    print("No previous checkpoints found on Google Drive. Training will start fresh.")

### Step 7: Launch YOLO11 Custom Model Training
Run `train.py` wrapper script. The robust training pipeline will dynamically manage VRAM, empty CUDA caches after every epoch, auto-retry on OOM, write logs, and synchronize weights/results to Google Drive in real-time.

In [ ]:
import sys
# Ensure repo path is in sys.path
if repo_path not in sys.path:
    sys.path.append(repo_path)

# Start the training process via CLI
# Set configuration overrides via argument parameters
epochs = 50
batch_size = 16 # Adjust depending on GPU (T4/L4: 16, A100: 32+)
img_size = 640

cmd = f"python train.py --epochs {epochs} --batch {batch_size} --imgsz {img_size}"
if resume_flag:
    cmd += " --resume"
else:
    cmd += " --auto-resume"

print(f"Executing training command: {cmd}")
!{cmd}

### Step 8: Helper Functions to Download Training Results
Provide convenient, easy-to-use functions to download `best.pt`, metrics charts, training logs, or the complete training directory.

In [ ]:
from google.colab import files

def download_best_model():
    """Download the best model weights file directly."""
    path = "/content/drive/MyDrive/LUMEN/checkpoints/best.pt"
    if os.path.exists(path):
        files.download(path)
        print("Initiated download for best.pt")
    else:
        print("ERROR: best.pt weights file not found on Google Drive checkpoints folder.")

def download_logs():
    """Download the training logs file."""
    path = "/content/drive/MyDrive/LUMEN/checkpoints/train.log"
    if os.path.exists(path):
        files.download(path)
        print("Initiated download for train.log")
    else:
        print("ERROR: train.log not found.")

def download_plots():
    """Download results plot charts and validation confusion matrices."""
    for filename in ["results.png", "confusion_matrix.png"]:
        path = f"/content/drive/MyDrive/LUMEN/checkpoints/{filename}"
        if os.path.exists(path):
            files.download(path)
            print(f"Initiated download for {filename}")
        else:
            print(f"ERROR: {filename} was not found on Google Drive checkpoints folder.")

def download_complete_training_zip():
    """Compresses and downloads the entire local training directory as a ZIP file."""
    local_run_dir = f"/content/{GITHUB_REPO}/runs/train_run"
    if os.path.exists(local_run_dir):
        zip_output = "/content/lumen_training_run_artifacts"
        print("Compressing local training run outputs to zip...")
        shutil.make_archive(zip_output, 'zip', local_run_dir)
        files.download(f"{zip_output}.zip")
        print("Initiated download for lumen_training_run_artifacts.zip")
    else:
        print("ERROR: Local runs directory not found.")

# To download, uncomment and run the desired functions below:
# download_best_model()
# download_logs()
# download_plots()
# download_complete_training_zip()